In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms
from PIL import Image, ImageFont, ImageDraw
import random
import string
import time
import copy

# Конфигурация
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Символы: цифры + латиница (верхний/нижний регистр)
CHARACTERS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
VOCAB = ['-'] + list(CHARACTERS)  # '-' - это blank-токен для CTC
CHAR_TO_IDX = {c: i for i, c in enumerate(VOCAB)}
IDX_TO_CHAR = {i: c for c, i in CHAR_TO_IDX.items()}
NUM_CLASSES = len(VOCAB)

# Параметры изображений
IMG_HEIGHT = 32
MAX_IMG_WIDTH = 256
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
FONT_PATH = '/content/drive/MyDrive/Sefer/centurygothic.ttf'  # ПУТЬ К НУЖНОМУ ШРИФТУ
IMG_DIR = '/content/drive/MyDrive/Sefer/cropped/train'         # Папка с реальными изображениями
CSV_PATH = '/content/drive/MyDrive/Sefer/cropped/ocr_annotations.csv'

Using device: cuda


In [ ]:
# Аугментация для обучения
train_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, MAX_IMG_WIDTH), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomAffine(degrees=(-5, 5), translate=(0.02, 0.02), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet stats
])

# Трансформация для валидации/инференса (без рандома)
val_transforms = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, MAX_IMG_WIDTH), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class RealOCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']
        if not fname.endswith(('.png', '.jpg', '.jpeg')):
            fname += '.png'
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        label = str(row['text'])
        if self.transform:
            img = self.transform(img)
        return img, label, img.shape[-1]  # Возвращаем ширину для падинга в collate


class SyntheticOCRDataset(Dataset):
    def __init__(self, font_path, num_samples, transform=None, max_len=10):
        self.font_path = font_path
        self.num_samples = num_samples
        self.transform = transform
        self.max_len = max_len
        self.chars = CHARACTERS

    def __len__(self):
        return self.num_samples

    def _render(self, text):
        font = ImageFont.truetype(self.font_path, IMG_HEIGHT - 6)
        bbox = font.getbbox(text)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        w = max(tw + 12, IMG_HEIGHT // 2)
        img = Image.new('RGB', (w, IMG_HEIGHT), (255, 255, 255))
        draw = ImageDraw.Draw(img)
        draw.text(((w - tw) // 2, (IMG_HEIGHT - th) // 2), text, font=font, fill=(0, 0, 0))
        return img

    def __getitem__(self, idx):
        length = random.randint(2, self.max_len)
        text = ''.join(random.choices(self.chars, k=length))
        img = self._render(text)
        if self.transform:
            img = self.transform(img)
        return img, text, img.shape[-1]

In [ ]:
def ocr_collate_fn(batch):
    images, labels, widths = zip(*batch)
    max_w = max(widths)

    # Паддинг изображений до максимальной ширины в батче
    padded_imgs = []
    for img, w in zip(images, widths):
        pad = max_w - w
        if pad > 0:
            img = F.pad(img, (0, pad, 0, 0), value=1.0)
        padded_imgs.append(img)
    images = torch.stack(padded_imgs)

    # Кодирование лейблов в СПИСОК тензоров (не склеиваем сразу!)
    targets_list = []
    target_lens = []
    for lbl in labels:
        idxs = [CHAR_TO_IDX[c] for c in lbl if c in CHAR_TO_IDX]
        targets_list.append(torch.tensor(idxs, dtype=torch.long))
        target_lens.append(len(idxs))

    target_lens = torch.tensor(target_lens, dtype=torch.long)
    # CNN сжимает ширину в 16 раз (4 maxpool 2x2). Все изображения теперь имеют width = max_w
    input_lens = torch.full((len(images),), max_w // 16, dtype=torch.long)

    # CTC требует: длина выходной последовательности >= длина целевой строки
    valid_mask = input_lens >= target_lens

    # Если вдруг весь батч оказался невалидным, возвращаем пустой батч
    if not valid_mask.any():
        return torch.zeros(0, 3, IMG_HEIGHT, max_w), torch.zeros(0, dtype=torch.long), \
               torch.zeros(0, dtype=torch.long), torch.zeros(0, dtype=torch.long)

    # Применяем маску к изображениям и длинам
    images = images[valid_mask]
    input_lens = input_lens[valid_mask]
    target_lens = target_lens[valid_mask]

    # Фильтруем список целей и только теперь конкатенируем
    filtered_targets = [t for t, v in zip(targets_list, valid_mask.tolist()) if v]
    targets = torch.cat(filtered_targets)

    return images, targets, input_lens, target_lens

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # CNN: экстракция признаков, сжатие высоты до 1
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.AdaptiveMaxPool2d((1, None))  # Гарантируем высоту 1
        )
        # RNN: BiLSTM для контекста символов
        self.rnn = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # x: (B, 3, H, W)
        conv = self.cnn(x)  # (B, 512, 1, W')
        b, c, h, w = conv.size()
        conv = conv.squeeze(2).permute(0, 2, 1)  # (B, W', 512)
        rnn_out, _ = self.rnn(conv)              # (B, W', 512)
        out = self.fc(rnn_out)                   # (B, W', num_classes)
        out = out.permute(1, 0, 2)               # (W', B, num_classes) для CTCLoss
        return torch.log_softmax(out, dim=2)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch_data in loader:
        if batch_data is None or batch_data[0].size(0) == 0:
            continue  # Пропускаем пустые батчи

        images, targets, input_lens, target_lens = batch_data
        images, targets = images.to(device), targets.to(device)
        input_lens, target_lens = input_lens.to(device), target_lens.to(device)

        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, targets, input_lens, target_lens)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step(total_loss / len(loader))
    return total_loss / len(loader)

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    for images, targets, input_lens, target_lens in loader:
        images, targets = images.to(device), targets.to(device)
        input_lens, target_lens = input_lens.to(device), target_lens.to(device)
        preds = model(images)
        loss = criterion(preds, targets, input_lens, target_lens)
        total_loss += loss.item()
    return total_loss / len(loader)

# Декодирование CTC
def decode_preds(preds):
    preds = preds.permute(1, 0, 2)  # (B, T, C)
    preds = preds.argmax(dim=2)     # (B, T)
    decoded = []
    for seq in preds:
        text, prev = [], -1
        for idx in seq:
            if idx != 0 and idx != prev:
                text.append(IDX_TO_CHAR[idx.item()])
            prev = idx
        decoded.append(''.join(text))
    return decoded

@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    for images, targets, input_lens, target_lens in loader:
        images = images.to(device)
        preds = model(images)
        decoded = decode_preds(preds)
        # Восстанавливаем оригинальные строки из targets
        gt_texts = []
        start = 0
        for tl in target_lens.tolist():
            gt_texts.append(''.join([IDX_TO_CHAR[i] for i in targets[start:start+tl].tolist()]))
            start += tl
        correct += sum(1 for d, g in zip(decoded, gt_texts) if d == g)
        total += len(gt_texts)
    return correct / total if total > 0 else 0.0

In [ ]:
# 1. Датасеты
real_ds = RealOCRDataset(CSV_PATH, IMG_DIR, train_transforms)
syn_ds = SyntheticOCRDataset(FONT_PATH, num_samples=len(real_ds)*2, transform=train_transforms) # 2:1 синтетика:реальные
train_ds = ConcatDataset([real_ds, syn_ds])

val_ds = RealOCRDataset(CSV_PATH, IMG_DIR, val_transforms) # Для примера берем те же данные, в реале split лучше сделать заранее

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=ocr_collate_fn, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=ocr_collate_fn, num_workers=4, pin_memory=True)

# 2. Модель и оптимизатор
model = CRNN(NUM_CLASSES).to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# 3. Цикл обучения
best_val_loss = float('inf')
best_model_state = None

print("🚀 Starting training...")
for epoch in range(EPOCHS):
    start = time.time()
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, DEVICE)
    val_loss = val_epoch(model, val_loader, criterion, DEVICE)
    val_acc = evaluate_accuracy(model, val_loader, DEVICE)
    elapsed = time.time() - start

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Time: {elapsed:.1f}s | LR: {optimizer.param_groups[0]['lr']:.1e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, 'best_ocr_crnn.pth')
        print("  💾 Saved best model!")

print("✅ Training finished.")
model.load_state_dict(torch.load('best_ocr_crnn.pth'))

SAVE_DIR = "/content/drive/MyDrive/Sefer/deploy/models/v1"
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(best_model_state, os.path.join(SAVE_DIR, "best_ocr_crnn.pth"))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


🚀 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


KeyboardInterrupt: 

In [ ]:
def predict_image(img_path, model, device):
    img = Image.open(img_path).convert('RGB')
    img = val_transforms(img)
    img = img.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pred = model(img)
    text = decode_preds(pred)[0]
    return text

# Пример использования:
result = predict_image('/content/drive/MyDrive/Sefer/cropped/all/MDZ088_1_objects_0_c0.82.png', model, DEVICE)
print(f"Распознанный текст: {result}")

Распознанный текст: 88


In [ ]:
result = predict_image('/content/drive/MyDrive/Sefer/cropped/all/TUL893_objects_0_c0.82.png', model, DEVICE)
print(f"Распознанный текст: {result}")

Распознанный текст: 893
